# Input fidelity audit and probability diagnostics

Show the archived 100-window input audit and inspect Brier, ECE and classification F1 for the main video probability configurations.

**Scope:** These notebooks recompute descriptive statistics from the archived data and predictions. They do not retrain the original encoders or produce new evaluation populations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().resolve()
if not (ROOT / "CITATION.cff").exists():
    ROOT = ROOT.parent
assert (ROOT / "CITATION.cff").exists(), "Run from the repository root or notebooks/"
print("Project:", ROOT.name)

## Audit results
Only sampled windows were rated. The audit does not certify correctness of every corpus label.

In [ ]:
import json
a=json.loads((ROOT / "results/input_audit/AUDIT_OCR_STT_SUMMARY.json").read_text())
print("Audit n:",a["n_total_sample"],"Window-label agreement:",a["b_agresi_window"])
rows=[]
for key in ["ocr_audit","stt_audit"]:
    x=a[key]; rows.append({"source":key,"n":x["n_evaluated"],"agreement":x["raw_agreement"],"weighted_kappa":x["weighted_kappa"],"faithful_n":x["consensus_distribution"].get("score_2_accurate",x["consensus_distribution"].get("score_2_faithful"))})
display(pd.DataFrame(rows))

## Keyframe reporting is item-level, not a new adjudication

In [ ]:
status=json.loads((ROOT / "results/input_audit/KEYFRAME_RECONCILIATION_STATUS.json").read_text())
print(status)
display(pd.read_csv(ROOT / "results/input_audit/KEYFRAME_ITEM_LEVEL_RECONCILIATION.csv").head())

## Window-weighted probability quality

In [ ]:
m=pd.read_csv(ROOT / "results/learned_fusion/TEST_COMPARISON_SEED42.csv")
chosen=["text","equal_vat","vat_lr"]
display(m[m.model.isin(chosen)][["model","macro_f1","brier","ece10","group_weighted_brier"]])
s=pd.read_csv(ROOT / "results/video/gold_adjudicated_v1.1/VIDEO_OOF_PREDICTIONS.csv")
s=s[s.seed==42]
for model,col in [("text","text_probability"),("equal_vat","equal_vat_probability")]:
    v=float(brier_score_loss(s.label_final,s[col])); reported=float(m.loc[m.model.eq(model),"brier"].iat[0]);print(model,"independently recomputed Brier",v,"archived",reported)
    assert abs(v-reported)<1e-10